# Laya run 3: envelope v2 — passages, neighbours, yes/no boundary (Kaggle, GPU T4 ×2)

Third run, on the new **`laya_v2`** dataset. Same labels, same lectures, same train / val / test split as runs 1 and 2.
What changes is **what Laya reads** (envelope v2, built by the app's own code, see `docs/laya-finetune/envelope-v2.md`):

* **Fixes in every v2 model:** the lecture fields come *before* the corpus text (a long hadith used to push the span out of
  the input), the duplicate verse text is gone, inputs go up to **2048 tokens** (nothing is silently cut), and the boundary is
  **two yes/no questions** — *was the start cut off?*, *was the end cut off?* — with the word-overlap evidence in the state.
  A "yes" weighs `NOUL_TRUE_WEIGHT` × in the loss (they are ~10 % of the answers).
* **The ablation**: what each new piece adds, everything else equal.

| run | consecutive verses as one passage (D) | nearest findings listed (A) | counterfactual boundary rows |
|---|---|---|---|
| `base` | – | – | – |
| `passages` | ✓ | – | – |
| `neighbours` | – | ✓ | – |
| `both` | ✓ | ✓ | – |
| `both_synth` | ✓ | ✓ | ✓ (train only) |

The **"before" line** is your current app model (`ZenoAbderrahim/laya-wahy-soft_asis`) on the v1 test file. Every model is
scored **per verse** (a passage's answer goes to each of its verses), so all lines are on exactly the same cards.

### What to read in the results
1. **Short verses** (`by context` table): *short (1-3 words) with a same-surah verse around* — the الواقعة 89 / عبس 35-36
   failure. The "real rejected" count there should drop to ~0 with D / A.
2. **No new false acceptances**: *not-a-quote recall* on your cards must not fall (A's risk: "has a neighbour → accept").
3. **Boundary**: balanced accuracy of *start cut off* / *end cut off*, against the no-model rule (0.51 on your cards).

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets:** `HF_TOKEN` (reads your private app model for the
  "before" line, and pushes the new models if `PUSH_TO_HUB`)
* **Input:** the **`laya_v2`** dataset (upload `data/laya_datagen/laya_v2.zip` as a new dataset, see the guide)
* **Run:** *Save Version → Save & Run All (Commit)*, browser can be closed. Expect **~3–3.5 h** (5 trainings at 2 epochs).
  `both` runs first, so a session that dies late still has the main answer.
* **Download afterwards:** only the `results/` folder of the version's Output (a few MB). Put it in
  `data/laya_datagen/results/run3/` and open `notebooks/results/laya_v2_compare.ipynb`.

## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q -U "laya>=0.4.12,<0.5" "transformers>=4.48.0" safetensors huggingface_hub pandas matplotlib tabulate
import laya, transformers
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Dataset, scripts, base checkpoint, Hub login
Stops early if the input is not `laya_v2` (envelope 2) or its scripts are older than this notebook.

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download, login
from laya.agent import _fix_tokenizer_config

DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v2")
manifest = json.loads((DATA / "manifest.json").read_text())
assert manifest["envelope"] == 2, "not an envelope-v2 dataset"
assert "--noul-true-weight" in (DATA / "train_ddp.py").read_text() and "def per_verse" in (DATA / "eval_laya.py").read_text(), (
    "Old scripts in the dataset: rebuild laya_v2.zip and upload it as a NEW VERSION of the dataset, then re-run.")
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest["created"], "| base model", manifest["base_model"],
      "| max_len", manifest["max_len"], "| variants", list(manifest["variants"]))
MODEL_ID = manifest["base_model"]          # convaiinnovations/laya-multilingual
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Hub: logged in")
except Exception as e:
    HF_TOKEN = None
    print("Hub: no HF_TOKEN secret, the 'before' line (private model) and the push will be skipped:", e)

## 4. Settings
Everything not listed here is run 2's recipe (effective batch 64, RLCD, the same learning rates, 2 epochs). The micro-batch
is 4 × 8 accumulation steps instead of 8 × 4 (same effective batch) because inputs can now be 2048 tokens long.

In [ ]:
RUNS = ["both", "base", "passages", "neighbours", "both_synth"]       # order: the main answer first
NOUL_TRUE_WEIGHT = 3.0      # a "yes, cut off" answer weighs 3x in the loss (~10 % of the yes/no answers are yes)
TRAIN_ARGS = (f"--epochs 2 --micro-batch 4 --grad-accum 8 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd "
              f"--max-len {manifest['max_len']} --head-max-len {manifest['head_max_len']} --noul-true-weight {NOUL_TRUE_WEIGHT}")
APP_FALSE_SHARE = 0.10      # the app's real stream is ~10 % "not a quote" (prior-corrected numbers)
BEFORE_MODEL = "ZenoAbderrahim/laya-wahy-soft_asis"     # the app's current model: the "before" line (v1 envelope)
PUSH_TO_HUB = True          # upload every new model to a PRIVATE Hub repo (needs the HF_TOKEN secret)
HUB_PREFIX = "ZenoAbderrahim/laya-wahy-v2"              # repo = f"{HUB_PREFIX}-{run}"

WORK = Path("/kaggle/working")
for d in ("results", "logs"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

def run(cmd, log):
    # Run a shell command, stream its output here and into a log file; returns the exit code.
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

import pandas as pd
train_false_share = {}
rows = []
for name in RUNS:
    v = manifest["variants"][name]
    tr = v["train"]
    train_false_share[name] = round(tr.get("verdict FALSE", 0) / tr["rows"], 4)
    rows.append({"run": name, "about": v["about"], "train rows": tr["rows"], "passages": tr.get("passages", 0),
                 "synthetic": tr.get("synthetic", 0), "start yes / items": f"{tr.get('start_missing true', 0)} / {tr.get('start_missing items', 0)}",
                 "end yes / items": f"{tr.get('end_missing true', 0)} / {tr.get('end_missing items', 0)}",
                 "val rows": v["val"]["rows"], "test rows": v["test"]["rows"], "test verses": v["test"]["verses (labelled)"]})
pd.DataFrame(rows).set_index("run")

## 5. The "before" line: your current app model on the v1 test file
Same cards, one row per verse, the v1 envelope it was trained on. Also scores the cached JEV / Gemini / base-Laya answers
and the shortcut baselines once (`baseline_*.json`).

In [ ]:
flags = f"--verse-flags {DATA}/test_verse_flags.json"
if HF_TOKEN:
    rc = run(f"python {DATA}/eval_laya.py predict --model {BEFORE_MODEL} --name 'before: soft_asis (app)' "
             f"--test {DATA}/v1_test.jsonl --out {WORK}/results/before_soft_asis.json --app-false-share {APP_FALSE_SHARE} "
             f"--batch-size 32 --baselines {DATA}/v1_baselines.json {flags}", WORK / "logs" / "eval_before.log")
    print("before:", "ok" if rc == 0 else f"failed (exit {rc}), see logs/eval_before.log")

## 6. Train + evaluate
Per run: `torchrun` trains on both GPUs (the checkpoint's config records `wahy_envelope: 2` and the run's options, so the
app knows what to send it), then `eval_laya.py` scores it on that variant's test file, per verse. The val line after each
epoch prints the yes/no boundary's **balanced** accuracy: it should climb well above 0.5.

In [ ]:
import time
status = {}
for name in RUNS:
    v = manifest["variants"][name]
    out = WORK / "models" / name
    t0 = time.time()
    try:
        cfg_extra = json.dumps(v["cfg_extra"]).replace('"', '\\"')
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {DATA}/{name}/train.jsonl --val {DATA}/{name}/val.jsonl --name laya-wahy-v2-{name} "
                 f"--cfg-extra \"{cfg_extra}\" {TRAIN_ARGS}", WORK / "logs" / f"train_{name}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[name] = f"training failed (exit {rc}), see logs/train_{name}.log"
            continue
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {name} --test {DATA}/{name}/test.jsonl "
                 f"--out {WORK}/results/{name}.json --train-false-share {train_false_share[name]} "
                 f"--app-false-share {APP_FALSE_SHARE} --batch-size 16 {flags}", WORK / "logs" / f"eval_{name}.log")
        status[name] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
        train_log = out / "train_log.json"
        if train_log.exists():
            (WORK / "results" / f"train_log_{name}.json").write_text(train_log.read_text())
    except Exception as e:
        status[name] = f"error: {e}"
    print(f"\n##### {name}: {status[name]}\n", flush=True)
json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Load the results (+ the no-model boundary rule)

In [ ]:
from IPython.display import Markdown, display
v1_rows = eval_laya.read_jsonl(DATA / "v1_test.jsonl")
vflags = json.loads((DATA / "test_verse_flags.json").read_text())
res = {n: json.load(open(WORK / "results" / f"{n}.json")) for n in RUNS if (WORK / "results" / f"{n}.json").exists()}
before = [json.load(open(WORK / "results" / "before_soft_asis.json"))] if (WORK / "results" / "before_soft_asis.json").exists() else []
rule = eval_laya.evaluate(*eval_laya.per_verse(v1_rows, eval_laya.rule_predictions(v1_rows), vflags), "word-overlap rule (no model)")
json.dump(rule, open(WORK / "results" / "rule_boundary.json", "w"), ensure_ascii=False)
teachers = [json.load(open(f)) for f in sorted(glob.glob(str(WORK / "results" / "baseline_*.json")))]
models = before + [res[n] for n in RUNS if n in res]
print("models:", [m["name"] for m in models], "| teacher baselines:", [t["name"] for t in teachers])

## 8. Verdict: overall, then the slices the new context is for
Columns as in runs 1–2. *real quotes wrongly rejected* is the costly error; *not-a-quote recall* must not fall.

In [ ]:
report = []
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    t = f"### verdict · {g}\n" + eval_laya.compare(models + teachers, g)
    report.append(t); display(Markdown(t))
for g in ["you", "you + Sonnet"]:
    for part in ("by context", "by span length"):
        t = f"### verdict {part} · {g}\n" + eval_laya.compare_slices(models, g, part)
        report.append(t); display(Markdown(t))
(WORK / "results" / "comparison.md").write_text("\n\n".join(report))

## 9. Boundary
**Balanced accuracy** is the column to read ("always complete" = 0.25 on 4 classes, 0.5 on a yes/no). The v2 models'
4-class answer is built from the two yes/no answers; *start / end: balanced / AUC* is the yes/no question itself.

In [ ]:
btext = []
for g in ["you", "written text", "you + Sonnet", "Sonnet · other lectures"]:
    t = f"### boundary · {g}\n" + eval_laya.compare_boundary([rule] + teachers + models, g)
    btext.append(t); display(Markdown(t))
(WORK / "results" / "boundary.md").write_text("\n\n".join(btext))

logs = {n: json.load(open(WORK / "results" / f"train_log_{n}.json")) for n in RUNS if (WORK / "results" / f"train_log_{n}.json").exists()}
display(pd.DataFrame([{"run": n, **{f"ep{e['epoch']} {k}": e["val"][k].get("balanced_accuracy", round(e["val"][k]["accuracy"], 4))
                                    for e in logs[n]["epochs"] for k in e["val"] if k != "all"}}
                      for n in logs]).set_index("run"))

## 10. One picture

In [ ]:
import matplotlib.pyplot as plt

def val(m, group, part, key, sub=None):
    x = ((m["groups"].get(group) or {}).get(part) or {})
    x = x.get(sub, {}) if sub else x
    x = x.get(key)
    return x[0] if isinstance(x, list) else x

def slice_rej(m, group, name):
    for sl in (m["groups"].get(group) or {}).get("verdict", {}).get("by context", []):
        if sl["slice"] == name:
            return sl["real quotes wrongly rejected"][0]

panels = [("you · AUC", lambda m: val(m, "you", "verdict", "AUC P(not a quote)")),
          ("you · real wrongly rejected", lambda m: val(m, "you", "verdict", "real quotes wrongly rejected")),
          ("you · not-a-quote recall", lambda m: val(m, "you", "verdict", "not-a-quote recall")),
          ("you+Sonnet · short verse next to its surah:\nreal wrongly rejected",
           lambda m: slice_rej(m, "you + Sonnet", "short (1-3 words) with a same-surah verse around")),
          ("you · boundary balanced acc", lambda m: val(m, "you", "boundary", "balanced accuracy"))]
names = [m["name"] for m in models]
fig, ax = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.8))
for a, (title, f) in zip(ax, panels):
    ys = [f(m) for m in models]
    a.bar(range(len(models)), [y if y is not None else 0 for y in ys], color=["#888"] + ["C0"] * (len(models) - 1))
    a.set_xticks(range(len(models)), names, rotation=45, ha="right", fontsize=8); a.set_title(title, fontsize=9); a.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(WORK / "results" / "ablation.png", dpi=120); plt.show()

## 11. (Optional) Push the models to the Hugging Face Hub — private
Every run that trained. In the app, choose the repo as the Laya checkpoint: its config tells the app to send envelope v2
with that run's options.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for n in RUNS:
        out = WORK / "models" / n
        if not (out / "model.safetensors").exists():
            continue
        repo = f"{HUB_PREFIX}-{n}"
        api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo, repo_type="model",
                          commit_message=f"laya-multilingual fine-tuned on Wahy laya_v2 ({n}, envelope v2, 2 epochs)")
        print("pushed", repo)